# Sesión 2 · Representación de la información I: Sparse vs Dense

La recuperación de información depende de cómo representamos consultas y documentos. Una representación léxica y una representación aprendida pueden recibir exactamente los mismos textos y producir rankings diferentes, porque preservan señales diferentes de relevancia.


## Preparación del entorno

La parte sparse/BM25 utiliza únicamente la biblioteca estándar de Python. La parte dense usa **LangChain como interfaz común** para modelos locales y servicios vía API. La práctica principal se realiza con **Qwen3-Embedding-0.6B**, un modelo abierto multilingüe y sensible a instrucciones.

> En Colab: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`. La primera descarga ronda 1,2 GB. En CPU funcionará, pero será sensiblemente más lenta.


In [ ]:
%pip install -qU \
    "langchain-core" \
    "langchain-huggingface" \
    "langchain-openai" \
    "langchain-google-genai" \
    "langchain-cohere" \
    "python-dotenv" \
    "sentence-transformers>=3.0" \
    "transformers>=4.51.0" \
    torch


### Imports y credenciales

Las claves se gestionan con `python-dotenv`. Crea un archivo `.env` en el directorio de trabajo con esta estructura:

```dotenv
OPENAI_API_KEY=...
GOOGLE_API_KEY=...
COHERE_API_KEY=...
```

El archivo `.env` contiene secretos y no debe incluirse en un repositorio.


In [1]:
import math
import os
import re
from collections import Counter
from pprint import pprint


In [2]:
from dotenv import load_dotenv
from langchain_cohere import CohereEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

load_dotenv()

variables_requeridas = ["OPENAI_API_KEY", "GOOGLE_API_KEY", "COHERE_API_KEY"]
variables_ausentes = [nombre for nombre in variables_requeridas if not os.getenv(nombre)]

if variables_ausentes:
    print("Faltan variables en .env:", ", ".join(variables_ausentes))
else:
    print("Credenciales cargadas desde .env.")


Credenciales cargadas desde .env.


---

# 1. El problema de representar información

## 1.1 Representar es elegir qué conservar

Un sistema de recuperación no compara directamente «significados». Primero transforma consultas y documentos en objetos que puede procesar: conjuntos de términos, vectores, estructuras o combinaciones de ellos.

Una representación es una función:

$$
r: \text{objeto original} \longrightarrow \text{objeto computable}
$$

Toda transformación introduce un **sesgo inductivo**: facilita detectar unas relaciones y dificulta otras. No existe una representación neutral ni universalmente mejor.

El mismo texto puede representarse, por ejemplo, como:

- una secuencia de caracteres;
- una secuencia de tokens;
- un conjunto o multiconjunto de palabras;
- un vector TF-IDF;
- uno o varios embeddings aprendidos;
- entidades, relaciones o metadatos.

La idea central es que cambiar la representación puede cambiar por completo qué documentos parecen cercanos.


### Una buena representación debería preservar… ¿qué?

La respuesta depende de la tarea. Para recuperación solemos valorar:

- **discriminación:** separar documentos relevantes de no relevantes;
- **invariancias útiles:** reconocer que una paráfrasis puede expresar la misma necesidad;
- **precisión léxica:** no perder códigos, nombres propios o terminología exacta;
- **comparabilidad:** permitir calcular una puntuación consulta–documento;
- **eficiencia:** poder indexar y buscar a escala;
- **robustez:** comportarse razonablemente ante ruido, idiomas y dominios distintos;
- **interpretabilidad:** entender por qué se obtuvo una puntuación, cuando sea necesario.

**Pregunta de comprensión.** Si buscamos jurisprudencia por número de sentencia, ¿qué propiedad priorizarías? ¿Y si buscamos preguntas equivalentes formuladas con vocabulario distinto?

<details><summary>Respuesta orientativa</summary>

El primer caso exige preservar cadenas exactas; el segundo, relaciones semánticas e invariancia a la paráfrasis. Una única representación puede no optimizar ambos objetivos.
</details>


## 1.2 Dos estrategias: sparse y dense

**Sparse (dispersa).** Las dimensiones suelen corresponder a elementos explícitos —normalmente términos del vocabulario— y la mayoría de componentes valen cero. La señal principal es léxica.

**Dense (densa).** Las dimensiones son aprendidas y casi todas contienen algún valor. La utilidad emerge de la geometría: productos escalares, cosenos o distancias reflejan relaciones aprendidas.

| Propiedad | Sparse | Dense |
|---|---|---|
| Coordenadas | Términos explícitos | Dimensiones aprendidas |
| Dimensión típica | Tamaño del vocabulario | Fija por el modelo |
| Valores no nulos | Muy pocos | Casi todos |
| Señal fuerte | Coincidencia léxica | Compatibilidad aprendida |
| Fortaleza intuitiva | Identificadores y términos exactos | Paráfrasis y variación lingüística |

No es una oposición entre «antiguo» y «moderno». BM25 sigue siendo una base extraordinariamente útil en sistemas reales; los embeddings resuelven problemas distintos.


---

# 2. Representaciones sparse y recuperación léxica

## 2.1 El vocabulario como sistema de coordenadas

Supongamos el vocabulario ordenado:

$$V=[\text{autonomía},\ \text{batería},\ \text{coche},\ \text{eléctrico},\ \text{precio}]$$

Cada posición tiene un significado explícito y estable. El documento «coche eléctrico con buena autonomía» puede convertirse en:

$$[1,0,1,1,0]$$

La posición no significa «grado de semanticidad»: cuenta la presencia o frecuencia del término asignado a esa coordenada.


In [3]:
def tokenizar(texto):
    """Tokenizador didáctico: minúsculas y palabras Unicode.

    No elimina stopwords ni aplica stemming: queremos poder observar
    claramente qué decisiones pertenecen a la representación.
    """
    return re.findall(r"\b\w+\b", texto.lower(), flags=re.UNICODE)

texto_ejemplo = "Coche eléctrico con buena autonomía."
tokens_ejemplo = tokenizar(texto_ejemplo)
print(tokens_ejemplo)


['coche', 'eléctrico', 'con', 'buena', 'autonomía']


**Experimento.** Cambia «eléctrico» por «eléctricos» o «autonomía» por «duración de la batería». Este tokenizador no sabe que ciertas formas están relacionadas: esa decisión tendría que introducirse mediante normalización, lematización o una representación diferente.

### Decisiones ocultas en una tokenización «sencilla»

Antes de crear vectores ya hemos decidido cómo tratar mayúsculas, puntuación, guiones, tildes, números, emojis o lenguas sin separación por espacios. En producción, la tokenización debe formar parte del diseño y de la evaluación; no es limpieza inocua.


## 2.2 Bag of Words (BoW)

Bag of Words representa un documento mediante la frecuencia de cada término y **descarta el orden**. Formalmente, para vocabulario $V$:

$$
\operatorname{BoW}(d)_j = \operatorname{count}(V_j,d)
$$

Construiremos primero el vocabulario y después una fila por documento.


In [4]:
documentos_bow = [
    "coche eléctrico con buena autonomía",
    "precio del coche eléctrico",
    "batería con autonomía y batería de repuesto",
]

def construir_vocabulario(documentos):
    return sorted({token for doc in documentos for token in tokenizar(doc)})

def vector_bow(texto, vocabulario):
    frecuencias = Counter(tokenizar(texto))
    return [frecuencias[termino] for termino in vocabulario]

vocabulario_bow = construir_vocabulario(documentos_bow)
matriz_bow = [vector_bow(doc, vocabulario_bow) for doc in documentos_bow]

print("Vocabulario:", vocabulario_bow)
for doc, vector in zip(documentos_bow, matriz_bow):
    print(f"{vector}  <- {doc}")


Vocabulario: ['autonomía', 'batería', 'buena', 'coche', 'con', 'de', 'del', 'eléctrico', 'precio', 'repuesto', 'y']
[1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0]  <- coche eléctrico con buena autonomía
[0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0]  <- precio del coche eléctrico
[1, 2, 0, 0, 1, 1, 0, 0, 0, 1, 1]  <- batería con autonomía y batería de repuesto


### Matriz documento–término y sparsity

Apilando los vectores obtenemos una matriz $X\in\mathbb{R}^{N\times |V|}$. En una colección real, cada documento utiliza una fracción mínima del vocabulario, por lo que casi todas las entradas son cero.

$$
\operatorname{sparsity}(X)=\frac{\#\{x_{ij}=0\}}{N\,|V|}
$$

Esta dispersión no es solo una característica conceptual: permite índices invertidos muy eficientes. Un índice no recorre todos los documentos; para cada término almacena qué documentos lo contienen.


In [5]:
def sparsity(matriz):
    total = sum(len(fila) for fila in matriz)
    ceros = sum(valor == 0 for fila in matriz for valor in fila)
    return ceros / total

print(f"Dimensiones: {len(matriz_bow)} documentos × {len(vocabulario_bow)} términos")
print(f"Sparsity: {sparsity(matriz_bow):.1%} de las celdas son cero")


Dimensiones: 3 documentos × 11 términos
Sparsity: 54.5% de las celdas son cero


### ¿Qué conserva y qué destruye BoW?

**Conserva:** identidad explícita de los términos y sus frecuencias.  
**Destruye:** orden, sintaxis, proximidad y gran parte del contexto.

Por ello, estas dos frases tienen exactamente el mismo vector:

- «el perro muerde al hombre»
- «el hombre muerde al perro»

Esto no convierte a BoW en inútil. Significa que es una representación adecuada cuando la señal que necesitamos depende sobre todo de *qué términos aparecen*, no de cómo se organizan.


In [6]:
par_a = "el perro muerde al hombre"
par_b = "el hombre muerde al perro"
vocabulario_par = construir_vocabulario([par_a, par_b])
print(vocabulario_par)
print(vector_bow(par_a, vocabulario_par))
print(vector_bow(par_b, vocabulario_par))
print("¿Son iguales?", vector_bow(par_a, vocabulario_par) == vector_bow(par_b, vocabulario_par))


['al', 'el', 'hombre', 'muerde', 'perro']
[1, 1, 1, 1, 1]
[1, 1, 1, 1, 1]
¿Son iguales? True


**Ejercicio.** Predice el vector de «coche coche batería» usando el vocabulario del primer ejemplo. Después compruébalo.

1. ¿Qué dimensión tendrá valor 2?
2. ¿Aparecerá una coordenada nueva?
3. ¿Qué ocurriría con una palabra fuera del vocabulario?


In [7]:
# Solución: ejecuta después de formular la predicción.
texto_prueba = "coche coche batería"
print(dict(zip(vocabulario_bow, vector_bow(texto_prueba, vocabulario_bow))))


{'autonomía': 0, 'batería': 1, 'buena': 0, 'coche': 2, 'con': 0, 'de': 0, 'del': 0, 'eléctrico': 0, 'precio': 0, 'repuesto': 0, 'y': 0}


## 2.3 No todos los términos informan igual

Las palabras muy comunes aparecen en muchos documentos y discriminan poco. Necesitamos separar:

- **TF (term frequency):** evidencia local; cuánto aparece el término en este documento.
- **DF (document frequency):** en cuántos documentos de la colección aparece.
- **IDF (inverse document frequency):** rareza global; más peso para términos presentes en pocos documentos.

Usaremos una versión suavizada:

$$
\operatorname{idf}(t)=\log\left(\frac{N+1}{\operatorname{df}(t)+1}\right)+1
$$

El suavizado evita divisiones problemáticas y mantiene pesos positivos. Existen otras variantes: lo importante es documentar y aplicar consistentemente la elegida.


In [8]:
corpus_tfidf = [
    "la batería del coche eléctrico mejora la autonomía",
    "el precio del coche eléctrico depende de la batería",
    "la autonomía real disminuye con el frío",
    "manual de mantenimiento del motor diésel",
]

def frecuencias_documento(texto):
    return Counter(tokenizar(texto))

def frecuencias_documentales(documentos):
    df = Counter()
    for doc in documentos:
        df.update(set(tokenizar(doc)))
    return df

def calcular_idf(documentos):
    n = len(documentos)
    df = frecuencias_documentales(documentos)
    return {t: math.log((n + 1) / (freq + 1)) + 1 for t, freq in df.items()}

idf = calcular_idf(corpus_tfidf)
for termino in sorted(idf, key=idf.get):
    print(f"{termino:15s} df={frecuencias_documentales(corpus_tfidf)[termino]}  idf={idf[termino]:.3f}")


la              df=3  idf=1.223
del             df=3  idf=1.223
batería         df=2  idf=1.511
eléctrico       df=2  idf=1.511
coche           df=2  idf=1.511
autonomía       df=2  idf=1.511
el              df=2  idf=1.511
de              df=2  idf=1.511
mejora          df=1  idf=1.916
precio          df=1  idf=1.916
depende         df=1  idf=1.916
frío            df=1  idf=1.916
real            df=1  idf=1.916
disminuye       df=1  idf=1.916
con             df=1  idf=1.916
motor           df=1  idf=1.916
manual          df=1  idf=1.916
diésel          df=1  idf=1.916
mantenimiento   df=1  idf=1.916


## 2.4 TF-IDF desde cero

Una definición sencilla es:

$$
\operatorname{tfidf}(t,d)=\operatorname{tf}(t,d)\cdot\operatorname{idf}(t)
$$

Aquí usamos TF bruto para que el mecanismo sea visible. Otras implementaciones pueden usar TF binario, logarítmico o normalizado. La similitud coseno permite comparar la **dirección** de los vectores reduciendo el efecto de su magnitud:

$$
\cos(\mathbf{q},\mathbf{d})=\frac{\mathbf{q}\cdot\mathbf{d}}{\|\mathbf{q}\|\|\mathbf{d}\|}
$$


In [9]:
def vector_tfidf(texto, vocabulario, idf):
    tf = frecuencias_documento(texto)
    return [tf[t] * idf.get(t, 0.0) for t in vocabulario]

def coseno(a, b):
    producto = sum(x * y for x, y in zip(a, b))
    norma_a = math.sqrt(sum(x * x for x in a))
    norma_b = math.sqrt(sum(y * y for y in b))
    return producto / (norma_a * norma_b) if norma_a and norma_b else 0.0

vocabulario_tfidf = construir_vocabulario(corpus_tfidf)
vectores_tfidf = [vector_tfidf(doc, vocabulario_tfidf, idf) for doc in corpus_tfidf]

consulta = "autonomía batería"
vector_consulta = vector_tfidf(consulta, vocabulario_tfidf, idf)
ranking_tfidf = sorted(
    [(coseno(vector_consulta, v), i, doc) for i, (v, doc) in enumerate(zip(vectores_tfidf, corpus_tfidf))],
    reverse=True,
)
for score, i, doc in ranking_tfidf:
    print(f"{score:.3f}  D{i+1}: {doc}")


0.474  D1: la batería del coche eléctrico mejora la autonomía
0.235  D3: la autonomía real disminuye con el frío
0.229  D2: el precio del coche eléctrico depende de la batería
0.000  D4: manual de mantenimiento del motor diésel


**Comprobación rápida.**

1. ¿Por qué «autonomía» pesa más que «la»?
2. ¿Puede TF-IDF recuperar una paráfrasis sin términos compartidos?
3. Si duplicamos literalmente todo el contenido de un documento, ¿qué mecanismos corrigen ese aumento?

<details><summary>Ideas clave</summary>

1. Aparece en menos documentos y tiene mayor IDF.  
2. No por sí solo: sin coincidencia léxica el producto escalar es cero.  
3. La similitud coseno reduce el efecto de escala, pero TF-IDF básico no modela explícitamente saturación ni longitud como lo hace BM25.
</details>

Esto motiva BM25: contar mejor no basta si la frecuencia crece sin rendimientos decrecientes o si la longitud altera las oportunidades de coincidencia.


---

# 3. BM25 y búsqueda basada en términos

## 3.1 De TF-IDF a BM25

BM25 conserva la lógica léxica e introduce dos comportamientos muy útiles:

1. **Saturación de frecuencia.** Las primeras apariciones aportan mucho; repetir veinte veces un término no lo hace veinte veces más relevante.
2. **Normalización por longitud.** Tres apariciones en 100 palabras no se interpretan igual que tres en 10 000.

BM25 produce directamente una puntuación consulta–documento; no necesitamos construir un embedding dense.


## 3.2 La fórmula, pieza a pieza

Para cada término $t$ de la consulta $q$:

$$
\operatorname{BM25}(q,d)=\sum_{t\in q}\operatorname{IDF}(t)\;
\frac{f(t,d)(k_1+1)}{f(t,d)+k_1\left(1-b+b\frac{|d|}{\operatorname{avgdl}}\right)}
$$

| Símbolo | Significado |
|---|---|
| $f(t,d)$ | frecuencia de $t$ en $d$ |
| $|d|$ | longitud del documento en tokens |
| $\operatorname{avgdl}$ | longitud media de la colección |
| $k_1$ | rapidez de saturación de TF (habitualmente en torno a 1,2–2,0) |
| $b$ | intensidad de la normalización por longitud (0: ninguna; 1: completa) |
| IDF | capacidad discriminativa global del término |

Usaremos la variante Robertson–Sparck Jones suavizada y no negativa:

$$
\operatorname{IDF}(t)=\log\left(1+\frac{N-df(t)+0.5}{df(t)+0.5}\right)
$$

Distintas bibliotecas implementan variantes ligeramente diferentes; por eso los scores absolutos no siempre coinciden. El ranking y la evaluación son lo verdaderamente importante.


In [10]:
def idf_bm25(termino, n_documentos, df):
    """IDF BM25 positiva y suavizada."""
    return math.log(1 + (n_documentos - df.get(termino, 0) + 0.5) /
                    (df.get(termino, 0) + 0.5))

def factor_tf_bm25(frecuencia, longitud_doc, longitud_media, k1=1.5, b=0.75):
    """Frecuencia saturada y corregida por longitud."""
    if frecuencia == 0:
        return 0.0
    normalizacion = 1 - b + b * longitud_doc / longitud_media
    return frecuencia * (k1 + 1) / (frecuencia + k1 * normalizacion)

print("Saturación para un documento de longitud media:")
for f in [0, 1, 2, 3, 5, 10, 20, 50]:
    print(f"f={f:2d} -> factor={factor_tf_bm25(f, 100, 100):.3f}")


Saturación para un documento de longitud media:
f= 0 -> factor=0.000
f= 1 -> factor=1.000
f= 2 -> factor=1.429
f= 3 -> factor=1.667
f= 5 -> factor=1.923
f=10 -> factor=2.174
f=20 -> factor=2.326
f=50 -> factor=2.427


El límite del factor cuando $f\to\infty$ es $k_1+1$. Con $k_1=1{,}5$, el factor se acerca a 2,5: crece, pero cada repetición añade menos. Observa ahora la longitud manteniendo la misma frecuencia.


In [11]:
print("Misma frecuencia (f=3), distinta longitud:")
for longitud in [50, 100, 200, 500]:
    factor = factor_tf_bm25(3, longitud, longitud_media=100, k1=1.5, b=0.75)
    print(f"longitud={longitud:3d} -> factor={factor:.3f}")


Misma frecuencia (f=3), distinta longitud:
longitud= 50 -> factor=1.905
longitud=100 -> factor=1.667
longitud=200 -> factor=1.333
longitud=500 -> factor=0.833


**Experimento.** Cambia `b` a 0 y repite la celda: la longitud deja de importar. Cambia `k1` y observa cómo cambia la curva de saturación. Los parámetros no crean señales nuevas; controlan cuánto pesan las ya explicadas.


## 3.3 Implementación completa desde cero

Separaremos el índice (estadísticas de la colección) del scoring. En producción usaríamos un motor optimizado e índice invertido; esta versión prioriza transparencia.


In [12]:
class BM25DesdeCero:
    def __init__(self, documentos, k1=1.5, b=0.75):
        self.documentos = documentos
        self.k1 = k1
        self.b = b
        self.tokens = [tokenizar(doc) for doc in documentos]
        self.tf = [Counter(tokens) for tokens in self.tokens]
        self.longitudes = [len(tokens) for tokens in self.tokens]
        self.avgdl = sum(self.longitudes) / len(self.longitudes)
        self.df = Counter()
        for tokens in self.tokens:
            self.df.update(set(tokens))

    def idf(self, termino):
        return idf_bm25(termino, len(self.documentos), self.df)

    def contribucion(self, termino, indice_doc):
        frecuencia = self.tf[indice_doc][termino]
        factor = factor_tf_bm25(
            frecuencia,
            self.longitudes[indice_doc],
            self.avgdl,
            self.k1,
            self.b,
        )
        return self.idf(termino) * factor

    def score(self, consulta, indice_doc):
        # set evita contar dos veces términos repetidos en la query.
        # Algunas variantes sí modelan la frecuencia en la consulta.
        return sum(self.contribucion(t, indice_doc) for t in set(tokenizar(consulta)))

    def explicar(self, consulta, indice_doc):
        return {
            t: self.contribucion(t, indice_doc)
            for t in sorted(set(tokenizar(consulta)))
        }

    def buscar(self, consulta, k=5):
        resultados = [
            (self.score(consulta, i), i, doc)
            for i, doc in enumerate(self.documentos)
        ]
        return sorted(resultados, key=lambda x: (-x[0], x[1]))[:k]


## 3.4 Corpus común para toda la comparación

A partir de aquí **no cambiaremos ni el corpus ni las consultas** al pasar de BM25 a dense. Así evitamos una comparación tramposa en la que cada método recibe un ejemplo favorable distinto.

El corpus mezcla soporte técnico, autonomía de portátiles y pagos. Incluye coincidencias exactas, paráfrasis y distractores con vocabulario superficialmente parecido.


In [13]:
corpus = [
    {"id": "D01", "texto": "Solucionar ERR_CONNECTION_RESET en Python al realizar peticiones HTTP."},
    {"id": "D02", "texto": "Diagnóstico de una conexión TCP cerrada inesperadamente por el servidor remoto."},
    {"id": "D03", "texto": "Guía de compra de una batería portátil externa para cargar el teléfono durante un viaje."},
    {"id": "D04", "texto": "Consejos para reducir el consumo energético y prolongar la autonomía del ordenador."},
    {"id": "D05", "texto": "Cómo calibrar la batería de un portátil cuando el porcentaje mostrado es incorrecto."},
    {"id": "D06", "texto": "Pasos para reclamar al banco un cobro repetido en la tarjeta y solicitar su devolución."},
    {"id": "D07", "texto": "Cancelar un cargo duplicado en tarjeta: formulario y plazos de la entidad."},
    {"id": "D08", "texto": "Motivos por los que una transferencia bancaria puede aparecer como pendiente."},
    {"id": "D09", "texto": "Errores habituales de conexión en aplicaciones web y cómo registrar trazas."},
]

consultas = {
    "exacta": "ERR_CONNECTION_RESET Python",
    "semantica": "cómo gastar menos batería en el portátil",
    "mixta": "cancelar cargo duplicado en tarjeta",
}

for item in corpus:
    print(f"{item['id']}: {item['texto']}")


D01: Solucionar ERR_CONNECTION_RESET en Python al realizar peticiones HTTP.
D02: Diagnóstico de una conexión TCP cerrada inesperadamente por el servidor remoto.
D03: Guía de compra de una batería portátil externa para cargar el teléfono durante un viaje.
D04: Consejos para reducir el consumo energético y prolongar la autonomía del ordenador.
D05: Cómo calibrar la batería de un portátil cuando el porcentaje mostrado es incorrecto.
D06: Pasos para reclamar al banco un cobro repetido en la tarjeta y solicitar su devolución.
D07: Cancelar un cargo duplicado en tarjeta: formulario y plazos de la entidad.
D08: Motivos por los que una transferencia bancaria puede aparecer como pendiente.
D09: Errores habituales de conexión en aplicaciones web y cómo registrar trazas.


In [14]:
textos_corpus = [item["texto"] for item in corpus]
bm25 = BM25DesdeCero(textos_corpus)

def imprimir_ranking_bm25(nombre, consulta, k=4):
    print(f"\n[{nombre.upper()}] {consulta!r}")
    for posicion, (score, i, texto) in enumerate(bm25.buscar(consulta, k), 1):
        print(f"{posicion}. {corpus[i]['id']}  score={score:.3f}  {texto}")

for nombre, consulta in consultas.items():
    imprimir_ranking_bm25(nombre, consulta)



[EXACTA] 'ERR_CONNECTION_RESET Python'
1. D01  score=4.464  Solucionar ERR_CONNECTION_RESET en Python al realizar peticiones HTTP.
2. D02  score=0.000  Diagnóstico de una conexión TCP cerrada inesperadamente por el servidor remoto.
3. D03  score=0.000  Guía de compra de una batería portátil externa para cargar el teléfono durante un viaje.
4. D04  score=0.000  Consejos para reducir el consumo energético y prolongar la autonomía del ordenador.

[SEMANTICA] 'cómo gastar menos batería en el portátil'
1. D05  score=4.778  Cómo calibrar la batería de un portátil cuando el porcentaje mostrado es incorrecto.
2. D03  score=3.210  Guía de compra de una batería portátil externa para cargar el teléfono durante un viaje.
3. D09  score=2.270  Errores habituales de conexión en aplicaciones web y cómo registrar trazas.
4. D01  score=0.939  Solucionar ERR_CONNECTION_RESET en Python al realizar peticiones HTTP.

[MIXTA] 'cancelar cargo duplicado en tarjeta'
1. D07  score=7.876  Cancelar un cargo dupli

### Explicar un score, no solo mostrarlo

BM25 es interpretable a nivel de términos: podemos inspeccionar cuánto aporta cada coincidencia. El score absoluto solo tiene sentido dentro de una consulta y una implementación; no es una probabilidad de relevancia.


In [15]:
consulta_mixta = consultas["mixta"]
for indice in [5, 6]:  # D06 y D07
    print(f"\n{corpus[indice]['id']}: {corpus[indice]['texto']}")
    for termino, aporte in bm25.explicar(consulta_mixta, indice).items():
        print(f"  {termino:10s} -> {aporte:.3f}")
    print(f"  TOTAL      -> {bm25.score(consulta_mixta, indice):.3f}")



D06: Pasos para reclamar al banco un cobro repetido en la tarjeta y solicitar su devolución.
  cancelar   -> 0.000
  cargo      -> 0.000
  duplicado  -> 0.000
  en         -> 0.718
  tarjeta    -> 1.246
  TOTAL      -> 1.964

D07: Cancelar un cargo duplicado en tarjeta: formulario y plazos de la entidad.
  cancelar   -> 1.897
  cargo      -> 1.897
  duplicado  -> 1.897
  en         -> 0.799
  tarjeta    -> 1.386
  TOTAL      -> 7.876


### ¿Cuándo es BM25 especialmente adecuado?

- códigos de error, SKU, DOI, matrículas, referencias legales;
- nombres propios y entidades raras;
- terminología especializada donde una palabra cambia el significado;
- búsqueda auditable donde necesitamos explicar coincidencias;
- dominios sin datos de entrenamiento o con vocabulario cambiante;
- primera etapa rápida y eficiente para recuperar candidatos.

**Ejercicio.** Añade un documento muy largo que repita «cargo» tres veces y otro corto que lo contenga una vez. Predice qué efecto tendrán `b=0` y `b=1`. Comprueba después creando dos índices.


---

# 4. Representaciones dense y embeddings

## 4.1 El lexical gap

La consulta y el documento relevante pueden expresar la misma necesidad sin compartir las palabras importantes:

> **Consulta:** «cómo gastar menos batería en el portátil»  
> **Documento:** «consejos para reducir el consumo energético y prolongar la autonomía del ordenador»

BM25 no dispone de una señal fuerte: *gastar menos* y *reducir consumo*, o *portátil* y *ordenador*, no son el mismo token. Este desfase se denomina **lexical gap**.

Podríamos añadir diccionarios, expansión de consultas o lematización. Los embeddings modernos ofrecen otra estrategia: aprender una geometría donde consulta y documento compatibles estén próximos aunque no sean lexicalmente idénticos.

La transformación puede resumirse como «texto → modelo → vector denso → posición en un espacio aprendido».


## 4.2 ¿Qué es un embedding moderno?

Un modelo aprende una función $f_\theta$ que transforma texto en un vector:

$$f_\theta(\text{texto})\in\mathbb{R}^d$$

- **Dense:** casi todas las componentes tienen valor.
- **Learned:** las coordenadas no fueron diseñadas como términos explícitos.
- **Geometric:** la utilidad está en las relaciones entre vectores.
- **Task-dependent:** el espacio refleja objetivos, datos, idiomas e instrucciones de entrenamiento; no es «semántica perfecta».

Las dimensiones individuales rara vez son interpretables. No conviene afirmar que «la dimensión 417 representa autonomía»: la información suele estar distribuida.


## 4.3 Intuición contrastiva para retrieval

Durante el entrenamiento se presentan parejas y negativos:

- pareja positiva $(q,d^+)$: queremos aumentar su similitud;
- documentos negativos $d^-$: queremos reducir su similitud respecto a $q$.

Una forma esquemática de pérdida es:

$$
\mathcal{L}=-\log \frac{e^{s(q,d^+)/\tau}}
{e^{s(q,d^+)/\tau}+\sum_j e^{s(q,d_j^-)/\tau}}
$$

No necesitamos memorizarla. La intuición es: el modelo no aprende una «posición correcta» aislada para cada texto; aprende **qué relaciones debe preservar la geometría**. La calidad de positivos, negativos y objetivo de entrenamiento importa enormemente.


## 4.4 Query y documento: espacio compatible, roles diferentes

Una consulta suele ser corta, incompleta y expresar una necesidad; un documento aporta contenido. Muchos modelos modernos distinguen esos roles mediante instrucciones, prefijos o un parámetro `input_type`.

- Qwen3-Embedding y multilingual-e5-large-instruct recomiendan añadir una instrucción a la **consulta**, no al documento.
- Cohere separa `search_query` y `search_document`.
- Algunos servicios admiten tipos de tarea específicos.

El objetivo no es que ambos textos sean idénticos, sino que el modelo aprenda cuándo un documento responde a una consulta.

**Pregunta rápida.** ¿Por qué sería arriesgado codificar documentos con la plantilla de query y consultas sin su instrucción? Porque estaríamos usando el modelo fuera del protocolo con el que fue entrenado/evaluado; el espacio puede seguir teniendo la misma dimensión, pero no necesariamente la geometría prevista.


## 4.5 Panorama práctico en 2026, sin convertirlo en catálogo

La decisión importante no es memorizar nombres, sino comprobar: idioma y dominio, objetivo de retrieval, longitud, dimensión, coste/latencia, privacidad y protocolo query/document.

Ejemplos relevantes en septiembre de 2026:

| Opción | Idea pedagógica relevante |
|---|---|
| **Qwen3-Embedding-0.6B** | abierto, 100+ idiomas, instrucciones, hasta 1024 dimensiones; práctica principal |
| **multilingual-e5-large-instruct** (Microsoft/intfloat) | multilingüe, consulta instruida, documentos sin instrucción; contexto de 512 tokens |
| **BGE-M3** (BAAI) | multilingüe y multi-función: dense, sparse y multi-vector |
| **OpenAI `text-embedding-3-large`** | servicio gestionado, fuerte en inglés y otros idiomas |
| **Google `gemini-embedding-001` / `gemini-embedding-2`** | texto; y opción multimodal en Embedding 2 según disponibilidad |
| **Cohere `embed-v4.0`** | `search_query`/`search_document`, multilingüe y multimodal |

Los identificadores y SDK cambian. Antes de producción, consulta siempre la documentación del proveedor; al final hay enlaces oficiales fechados.


## 4.6 Dos alternativas abiertas para contrastar el protocolo

Estos ejemplos permiten comparar **cómo se usa** cada familia:

- `multilingual-e5-large-instruct`: consulta con instrucción; documentos sin instrucción.
- `BAAI/bge-m3`: puede usarse como encoder dense con Sentence Transformers, aunque su propuesta completa también produce señales sparse y multi-vector.

La arquitectura del recuperador (codificar documentos una vez, codificar la query y ordenar por similitud) permanece igual.
 
Para sustituir el modelo principal basta con cambiar `model_name` manteniendo la interfaz de LangChain:

```python
e5 = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large-instruct",
    encode_kwargs={"normalize_embeddings": True},
)
bge = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    encode_kwargs={"normalize_embeddings": True},
)
```

En E5 debe conservarse la plantilla instruida para la consulta; BGE-M3 puede utilizarse como encoder dense directamente. Las celdas siguientes no cargan estas alternativas para evitar descargas redundantes.


## 4.7 Práctica principal: Qwen3-Embedding-0.6B

Usaremos la integración `HuggingFaceEmbeddings` de LangChain. Al ejecutar la siguiente celda por primera vez se descargarán los pesos. La instrucción describe la tarea de recuperación y se añade solo a las consultas, de acuerdo con la tarjeta del modelo.


In [16]:
MODELO_DENSE = "Qwen/Qwen3-Embedding-0.6B"
INSTRUCCION = (
    "Given a Spanish web search query, retrieve relevant passages that answer the query"
)
modelo_dense = HuggingFaceEmbeddings(
    model_name=MODELO_DENSE,
    encode_kwargs={"normalize_embeddings": True},
)
print("Modelo cargado mediante LangChain:", MODELO_DENSE)


/Users/aruizart/Documents/01. Trabajo/01. EBIS/02. Máster en  Ingeniería de Automatización con IA Agéntica/05. Sistemas de recuperación de información, RAG y Langchain/Sesión 02/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5194.89it/s]


Modelo cargado mediante LangChain: Qwen/Qwen3-Embedding-0.6B


In [17]:
def preparar_query_qwen(consulta):
    return f"Instruct: {INSTRUCCION}\nQuery: {consulta}"

# Documentos: se codifican una sola vez y podrían almacenarse en un índice vectorial.
embeddings_documentos = modelo_dense.embed_documents(textos_corpus)
print(f"Matriz de documentos: {len(embeddings_documentos)} × {len(embeddings_documentos[0])}")
print("Primeras 8 componentes de D01:", embeddings_documentos[0][:8])


Matriz de documentos: 9 × 1024
Primeras 8 componentes de D01: [0.034912109375, 0.10009765625, -0.00775146484375, -0.047607421875, 0.062255859375, -0.01324462890625, -0.041259765625, -0.007781982421875]


In [18]:
def buscar_dense(consulta, k=5):
    embedding_query = modelo_dense.embed_query(preparar_query_qwen(consulta))
    scores = [coseno(embedding_query, vector_doc) for vector_doc in embeddings_documentos]
    orden = sorted(range(len(scores)), key=lambda i: (-float(scores[i]), i))[:k]
    return [(float(scores[i]), i, textos_corpus[i]) for i in orden]

for nombre, consulta in consultas.items():
    print(f"\n[{nombre.upper()}] {consulta!r}")
    for posicion, (score, i, texto) in enumerate(buscar_dense(consulta, 4), 1):
        print(f"{posicion}. {corpus[i]['id']}  cos={score:.3f}  {texto}")



[EXACTA] 'ERR_CONNECTION_RESET Python'
1. D01  cos=0.892  Solucionar ERR_CONNECTION_RESET en Python al realizar peticiones HTTP.
2. D02  cos=0.546  Diagnóstico de una conexión TCP cerrada inesperadamente por el servidor remoto.
3. D09  cos=0.440  Errores habituales de conexión en aplicaciones web y cómo registrar trazas.
4. D07  cos=0.166  Cancelar un cargo duplicado en tarjeta: formulario y plazos de la entidad.

[SEMANTICA] 'cómo gastar menos batería en el portátil'
1. D03  cos=0.571  Guía de compra de una batería portátil externa para cargar el teléfono durante un viaje.
2. D05  cos=0.560  Cómo calibrar la batería de un portátil cuando el porcentaje mostrado es incorrecto.
3. D04  cos=0.543  Consejos para reducir el consumo energético y prolongar la autonomía del ordenador.
4. D07  cos=0.288  Cancelar un cargo duplicado en tarjeta: formulario y plazos de la entidad.

[MIXTA] 'cancelar cargo duplicado en tarjeta'
1. D07  cos=0.826  Cancelar un cargo duplicado en tarjeta: formulario 

### Qué observar al ejecutar

- Los vectores tienen muchas componentes no nulas, pero imprimirlas todas no ayuda a interpretar el modelo.
- La consulta semántica debería elevar D04 pese al escaso solapamiento léxico.
- El identificador exacto D01 puede recuperarse bien, pero dense no ofrece la misma garantía intuitiva de coincidencia exacta que BM25.
- Los scores coseno son relativos al modelo y a esta consulta: no son probabilidades.

**Pequeño experimento.** Ejecuta la query semántica sin instrucción y compara el orden. Una única prueba no demuestra superioridad, pero hace visible que el protocolo de entrada forma parte del modelo.


## 4.8 Modelos vía API con la interfaz de LangChain

Usaremos siempre la abstracción común `Embeddings` de LangChain:

- `embed_documents(lista_de_textos)` para indexar documentos;
- `embed_query(consulta)` para codificar la necesidad de información.

Esto permite cambiar de proveedor sin reescribir la lógica de scoring. Las integraciones conservan las particularidades necesarias de cada API, como los roles query/document.

### OpenAI

`text-embedding-3-large` sigue documentado como el modelo de embeddings más capaz de OpenAI para tareas en inglés y otros idiomas. Para una alternativa más económica puede evaluarse `text-embedding-3-small`.


In [19]:
embeddings_openai = OpenAIEmbeddings(model="text-embedding-3-large")
print("Conector LangChain preparado:", type(embeddings_openai).__name__)


Conector LangChain preparado: OpenAIEmbeddings


### Google Gemini API

Para texto, `gemini-embedding-001` permanece disponible; `gemini-embedding-2` añade un espacio multimodal para texto, imágenes, audio, vídeo y documentos. La disponibilidad exacta puede depender del producto/región.


In [20]:
embeddings_google = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
print("Conector LangChain preparado:", type(embeddings_google).__name__)


Conector LangChain preparado: GoogleGenerativeAIEmbeddings


### Cohere

Cohere hace explícita la asimetría mediante `input_type="search_query"` y `input_type="search_document"`.


In [21]:
embeddings_cohere = CohereEmbeddings(model="embed-v4.0")
print("Conector LangChain preparado:", type(embeddings_cohere).__name__)


Conector LangChain preparado: CohereEmbeddings


### Una única función de ranking para cualquier proveedor

La ventaja pedagógica de LangChain aparece aquí: la función siguiente no contiene ningún `if` por proveedor. Solo exige un objeto que implemente la interfaz `Embeddings`.


In [22]:
def buscar_con_langchain(modelo_embeddings, consulta, documentos, k=4):
    vectores_documentos = modelo_embeddings.embed_documents(documentos)
    vector_consulta = modelo_embeddings.embed_query(consulta)
    scores = [coseno(vector_consulta, vector_doc) for vector_doc in vectores_documentos]
    orden = sorted(range(len(scores)), key=lambda i: (-scores[i], i))[:k]
    return [(scores[i], i, documentos[i]) for i in orden]

proveedores_api = {
    "OpenAI": embeddings_openai,
    "Google": embeddings_google,
    "Cohere": embeddings_cohere,
}

for proveedor, embeddings in proveedores_api.items():
    print(f"\n{proveedor} — {consultas['semantica']!r}")
    for posicion, (score, i, texto) in enumerate(
        buscar_con_langchain(embeddings, consultas["semantica"], textos_corpus), 1
    ):
        print(f"{posicion}. {corpus[i]['id']}  cos={score:.3f}  {texto}")



OpenAI — 'cómo gastar menos batería en el portátil'
1. D04  cos=0.673  Consejos para reducir el consumo energético y prolongar la autonomía del ordenador.
2. D05  cos=0.543  Cómo calibrar la batería de un portátil cuando el porcentaje mostrado es incorrecto.
3. D03  cos=0.485  Guía de compra de una batería portátil externa para cargar el teléfono durante un viaje.
4. D06  cos=0.276  Pasos para reclamar al banco un cobro repetido en la tarjeta y solicitar su devolución.

Google — 'cómo gastar menos batería en el portátil'
1. D04  cos=0.803  Consejos para reducir el consumo energético y prolongar la autonomía del ordenador.
2. D05  cos=0.738  Cómo calibrar la batería de un portátil cuando el porcentaje mostrado es incorrecto.
3. D03  cos=0.687  Guía de compra de una batería portátil externa para cargar el teléfono durante un viaje.
4. D09  cos=0.618  Errores habituales de conexión en aplicaciones web y cómo registrar trazas.

Cohere — 'cómo gastar menos batería en el portátil'
1. D04  c

---

# 5. Sparse vs Dense: mismo corpus, mismas consultas

## 5.1 Comparación controlada

No basta con contar una anécdota donde cada método recibe una query diferente. Aquí ambos sistemas reciben los mismos nueve documentos y estas tres consultas:

| Caso | Consulta | Qué pone a prueba | Relevancia esperada |
|---|---|---|---|
| Exacta | `ERR_CONNECTION_RESET Python` | identificador y lenguaje exacto | D01 |
| Semántica | `cómo gastar menos batería en el portátil` | lexical gap | D04 |
| Mixta | `cancelar cargo duplicado en tarjeta` | coincidencia exacta + paráfrasis | D07 y D06 |

No buscamos proclamar un ganador universal, sino diagnosticar **qué señal** utiliza cada ranking.


In [23]:
def ids_bm25(consulta, k=4):
    return [corpus[i]["id"] for _, i, _ in bm25.buscar(consulta, k)]

def ids_dense(consulta, k=4):
    return [corpus[i]["id"] for _, i, _ in buscar_dense(consulta, k)]

print(f"{'CASO':10s} {'BM25 TOP-4':30s} {'DENSE TOP-4'}")
print("-" * 75)
for nombre, consulta in consultas.items():
    sparse = ", ".join(ids_bm25(consulta))
    dense = ", ".join(ids_dense(consulta))
    print(f"{nombre:10s} {sparse:30s} {dense}")


CASO       BM25 TOP-4                     DENSE TOP-4
---------------------------------------------------------------------------
exacta     D01, D02, D03, D04             D01, D02, D09, D07
semantica  D05, D03, D09, D01             D03, D05, D04, D07
mixta      D07, D06, D01, D09             D07, D06, D08, D09


## 5.2 Leer los tres casos

### Caso A — donde la señal sparse debería ganar claramente

`ERR_CONNECTION_RESET` es una cadena altamente discriminativa. BM25 puede explicar el resultado por coincidencia exacta e IDF elevada. Dense puede reconocer el tema de errores de red, pero esa amplitud puede acercar D02 o D09. Cuando importa el identificador literal, la precisión léxica es difícil de sustituir.

### Caso B — donde dense debería ganar claramente

La query habla de «gastar menos batería» y D04 de «reducir consumo energético» y «prolongar autonomía». BM25 puede verse atraído por D03 o D05 porque contienen *batería* y *portátil*. Dense tiene la oportunidad de cerrar el lexical gap y situar D04 primero.

### Caso C — mixto/ambiguo

D07 coincide casi literalmente con la consulta. D06 expresa el procedimiento relevante mediante *cobro repetido* y *reclamar/devolución*. BM25 aporta precisión sobre D07; dense puede reconocer D06 como otra respuesta válida. Ambos aportan candidatos útiles por razones diferentes.

> Los resultados concretos pueden variar con versión del modelo, hardware y preprocesamiento. Eso no invalida la práctica: obliga a observar, formular hipótesis y evaluar.


## 5.3 Evaluación mínima: no confundir una demo con evidencia

Definimos juicios de relevancia pequeños para calcular `Recall@k`. En un proyecto real necesitaríamos muchas consultas representativas, evaluadores consistentes y métricas como Recall@k, MRR o nDCG.


In [24]:
relevantes = {
    "exacta": {"D01"},
    "semantica": {"D04"},
    "mixta": {"D06", "D07"},
}

def recall_a_k(ranking_ids, relevantes_ids, k):
    recuperados = set(ranking_ids[:k])
    return len(recuperados & relevantes_ids) / len(relevantes_ids)

for nombre, consulta in consultas.items():
    r_bm25 = recall_a_k(ids_bm25(consulta, 3), relevantes[nombre], 3)
    r_dense = recall_a_k(ids_dense(consulta, 3), relevantes[nombre], 3)
    print(f"{nombre:10s} Recall@3 BM25={r_bm25:.2f}  Dense={r_dense:.2f}")


exacta     Recall@3 BM25=1.00  Dense=1.00
semantica  Recall@3 BM25=0.00  Dense=1.00
mixta      Recall@3 BM25=1.00  Dense=1.00


### Ejercicio de análisis

Analiza un ranking inesperado sin recurrir a «el modelo es malo»:

1. ¿Qué tokens exactos explican BM25?
2. ¿Qué relación aprendida podría explicar dense?
3. ¿El juicio de relevancia es único o hay varias intenciones posibles?
4. ¿Cambiaría la conclusión en un corpus de un millón de documentos?
5. ¿Qué query adicional usaríais para falsar vuestra hipótesis?

**Ejercicio adicional.** Añade cinco documentos difíciles: un identificador parecido, una negación, una paráfrasis, un documento bilingüe y otro muy largo. Evalúa qué errores aparecen.


## 5.4 Cierre: complementariedad, no competición

**Sparse preserva señal léxica explícita.**  
**Dense preserva relaciones aprendidas en una geometría.**

Ninguno domina todas las necesidades:

- exactitud terminológica e interpretabilidad favorecen sparse;
- paráfrasis y variabilidad lingüística favorecen dense;
- muchas consultas reales mezclan ambas cosas.

La conclusión operativa es sencilla:

> **Misma consulta + mismo corpus + distinta representación = distinta señal de relevancia y, potencialmente, distinto ranking.**

El siguiente paso consiste en combinar ambas señales mediante **hybrid retrieval**, fusionar rankings y aplicar **reranking** sobre los candidatos recuperados.


## Resumen de estudio

1. Representar es comprimir y elegir: siempre se conserva algo y se pierde algo.
2. BoW utiliza el vocabulario como coordenadas y descarta el orden.
3. TF-IDF combina evidencia local (TF) y rareza global (IDF).
4. BM25 añade saturación de frecuencia y normalización por longitud.
5. Un embedding moderno es un vector dense cuya geometría se aprende.
6. Query y documento pueden requerir protocolos de codificación distintos.
7. Sparse destaca en coincidencias exactas; dense puede cerrar el lexical gap.
8. La comparación justa usa mismo corpus, mismas consultas y juicios explícitos.
9. Los scores no son probabilidades y no se comparan alegremente entre queries/modelos.
10. Los sistemas modernos suelen combinar señales y refinar candidatos.


## Fuentes y documentación (consultadas en septiembre de 2026)

### Modelos abiertos

- [Qwen3-Embedding-0.6B — tarjeta oficial en Hugging Face](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B)
- [multilingual-e5-large-instruct — tarjeta del modelo](https://huggingface.co/intfloat/multilingual-e5-large-instruct)
- [BGE-M3 — tarjeta oficial de BAAI](https://huggingface.co/BAAI/bge-m3)

### APIs gestionadas

- [LangChain — interfaz e integraciones de embeddings](https://docs.langchain.com/oss/python/integrations/embeddings)
- [LangChain — construcción de búsqueda semántica](https://docs.langchain.com/oss/python/langchain/knowledge-base)
- [LangChain — integración de CohereEmbeddings](https://docs.langchain.com/oss/python/integrations/embeddings/cohere)
- [OpenAI — guía de embeddings](https://developers.openai.com/api/docs/guides/embeddings)
- [OpenAI — text-embedding-3-large](https://developers.openai.com/api/docs/models/text-embedding-3-large)
- [Google — Gemini Embeddings](https://ai.google.dev/gemini-api/docs/embeddings)
- [Cohere — Introduction to Embeddings](https://docs.cohere.com/docs/embeddings)

### Para profundizar

- Robertson, S. y Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond*.
- Wang et al. (2024). *Multilingual E5 Text Embeddings: A Technical Report*.
- Qwen Team (2025). *Qwen3 Embedding: Advancing Text Embedding and Reranking Through Foundation Models*.

> **Nota de reproducibilidad.** Los nombres de modelos, SDK y disponibilidad pueden cambiar. El notebook separa los principios estables de los ejemplos de proveedor y fecha estos últimos deliberadamente.
